# Project - Airline AI Assistant

In [2]:
# Imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI 
import gradio as gr
import sqlite3


In [3]:
# Initialization

load_dotenv(override = True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else: 
    print("OpenAI API Key not set")

MODEL = "gpt-4.1-mini" 
openai = OpenAI() 

DB = "prices.db"

OpenAI API Key exists and begins sk-proj-


In [4]:
system_message = """ 
You are a helpful assistant for an Airline called FlightAI. 
Give short, courteous answers, no more than 1 sentence. Always
be accurate. If you don't know the answer, say so.
"""

In [7]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}",
          flush = True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?',
                       (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]:.2f}" if result else "No price data is available for this city"


In [8]:
get_ticket_price("london")

DATABASE TOOL CALLED: Getting price for london


'Ticket price to london is $799.00'

In [9]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", 
          "function": price_function}]
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

In [10]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, 
                                              messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat,
                 type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [11]:
def chat(message, history):
    history = [{"role":h["role"], 
               "content":h["content"]} for h in history]
    messages = [{"role": "system", 
                 "content": system_message}] + history + [{"role": "user",
                                                           "content": message}]
    response = openai.chat.completions.create(model = MODEL,
                                              messages = messages,
                                              tools = tools)
    
    while response.choices[0].finish_reason =="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model = MODEL,
                                                  messages = messages,
                                                  tools = tools)
    return response.choices[0].message.content
